# Play Clean Up

Run the final cell and enter one numeric action for each prompted agent. After `player_n` is entered, the complete action dictionary executes immediately as one parallel step.

| Key | Action |
|---:|---|
| `0` | no-op |
| `1` | move forward |
| `2` | strafe right |
| `3` | strafe left |
| `4` | move backward |
| `5` | turn left |
| `6` | turn right |
| `7` | zap beam |
| `8` | cleaning beam |

Press **Backspace** to reset the round and environment, or **Escape** to quit.

In [ ]:
from masa.envs.multiagent.tabular import CleanUp

SEED = 0
WINDOW_SIZE = 960


In [ ]:
def play(seed=SEED):
    import pygame

    key_to_action = {
        pygame.K_0: 0, pygame.K_KP0: 0,
        pygame.K_1: 1, pygame.K_KP1: 1,
        pygame.K_2: 2, pygame.K_KP2: 2,
        pygame.K_3: 3, pygame.K_KP3: 3,
        pygame.K_4: 4, pygame.K_KP4: 4,
        pygame.K_5: 5, pygame.K_KP5: 5,
        pygame.K_6: 6, pygame.K_KP6: 6,
        pygame.K_7: 7, pygame.K_KP7: 7,
        pygame.K_8: 8, pygame.K_KP8: 8,
    }
    action_names = {
        0: "no-op", 1: "forward", 2: "strafe right",
        3: "strafe left", 4: "backward", 5: "turn left",
        6: "turn right", 7: "zap beam", 8: "cleaning beam",
    }
    env = CleanUp(render_mode="human", render_window_size=WINDOW_SIZE)
    observations, infos = env.reset(seed=seed)
    pending = {}
    round_number = 1
    clock = pygame.time.Clock()

    def announce_next():
        agent = env.agents[len(pending)]
        pygame.display.set_caption(f"MASA - Clean Up | Round {round_number} | Input: {agent}")
        print(f"Round {round_number}: enter action 0-8 for {agent}")

    announce_next()
    try:
        running = True
        while running and not env.human_window_closed:
            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key == pygame.K_ESCAPE:
                    running = False
                    break
                if event.key == pygame.K_BACKSPACE:
                    observations, infos = env.reset(seed=seed)
                    pending.clear()
                    round_number = 1
                    print("Environment reset.")
                    announce_next()
                    continue
                if event.key not in key_to_action:
                    continue

                agent = env.agents[len(pending)]
                action = key_to_action[event.key]
                pending[agent] = action
                print(f"  {agent}: {action} ({action_names[action]})")
                if len(pending) < len(env.agents):
                    announce_next()
                    continue

                actions = dict(pending)
                pending.clear()
                observations, rewards, terminations, truncations, infos = env.step(actions)
                print("  simultaneous step -> rewards:", rewards)
                unsafe = [agent for agent, obs in observations.items() if env.cost_fn(env.label_fn(obs))]
                if unsafe:
                    print("  unsafe agents:", unsafe)
                if any(terminations.values()) or any(truncations.values()):
                    print("Episode finished; resetting.")
                    observations, infos = env.reset()
                    round_number = 1
                else:
                    round_number += 1
                announce_next()

            if running:
                env.render()
                clock.tick(30)
    finally:
        env.close()

play()
